# Vizualizacija podatkov — B: vaja (telekom)
**Akademija UI za poslovne aplikacije — Modul 1, Vizualizacija podatkov: kako izbrati pravi graf?**

Predavatelj je v zvezku `A_vizualizacija` grafe pokazal na spletni trgovini. Zdaj jih narišete na podatkih telekoma.

### Ponovitev
Graf izberemo glede na **vprašanje** in **tip spremenljivk**:

| Vprašanje | Spremenljivke | Graf |
|---|---|---|
| Kako so vrednosti porazdeljene? | 1 numerična | histogram + KDE, škatla z brki |
| Kako pogoste so kategorije? | 1 kategorična | stolpični diagram |
| Ali sta povezani? | 2 numerični | razsevni diagram |
| Kako se spreminja v času? | čas + numerična | črtni graf |
| Ali se skupine razlikujejo? | kategorična + numerična | škatle / violine po skupinah |
| Kaj je povezano s čim? | več numeričnih | korelacijska matrika; barva in podgrafi |

Grafe rišemo s knjižnico **plotnine** — "slovnica grafov", enako kot ggplot2 v R.
Graf sestavimo iz **plasti**, ki jih seštejemo s `+`:
- `ggplot(tabela, aes(x=..., y=...))` — podatki in kateri stolpec gre na katero os (ali v barvo: `fill=`, `color=`),
- `geom_...()` — oblika: `geom_histogram`, `geom_boxplot`, `geom_col`, `geom_point`, `geom_line` …,
- `scale_...`, `coord_...`, `facet_...`, `labs(...)`, `theme(...)` — merila osi, podgrafi, naslovi, videz.

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu. Koda je skrita — odprete jo s klikom na celico.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

### Knjižnice in videz grafov
Funkcije `histogram_in_skatla`, `graf_pogostosti` in `graf_korelacij` smo napisali v zvezku `A_vizualizacija`;
tu jih uvozimo iz `utils/orodja.py`.

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, BARVE_ODHOD
from utils.orodja import histogram_in_skatla, graf_pogostosti, graf_korelacij
nastavi_slog()      # enoten videz grafov (koda je v zvezku A_vizualizacija)

---
# Del B — Vaja: telekom
Na vrhu celice zamenjate ime stolpca (med narekovaji) in celico ponovno zaženete (**Shift + Enter**).
Graf se pokaže, če je v zadnji vrstici celice; drugje ga pokažemo z `.show()`.

In [ ]:
telekom = pd.read_csv(DATA / "telekom.csv")
list(telekom.columns)

## B1. Porazdelitev ene numerične spremenljivke
Poskusite: `"cena_naprave"`, `"mesecev_narocnine"`, `"povp_minute"`, `"dni_z_napravo"`.
Pri stolpcih z dolgim repom nastavite `LOG = True`.

**Vprašanje:** Ima porazdelitev en vrh ali več? Kje je glavnina strank? Je dolg rep?

In [ ]:
STOLPEC = "povp_minute"      # ← zamenjajte
LOG = True                   # ← True = logaritemska os

histogram_in_skatla(telekom, STOLPEC, log=LOG).show()
telekom[STOLPEC].describe()

<details>
<summary><b>Rešitev (klikni)</b></summary>

- `cena_naprave` ima **več ločenih vrhov**: telefoni se prodajajo po nekaj fiksnih cenah (10, 30, 60, 80, 100, 130, 150, 200 …).
  KDE krivulja to zgladi — zato vedno poglejte tudi stolpce histograma.
- `povp_minute` ima **dolg desni rep** — z `LOG = True` je slika veliko bolj berljiva.
- `mesecev_narocnine` se začne pri 6 — v podatkih so samo stranke z vsaj 6 meseci naročnine.

</details>

Preverimo: najpogostejše cene naprav (zaokrožene na cel USD):

In [ ]:
telekom["cena_naprave"].round().value_counts().head(8)

## B2. Pogostost kategorij
`graf_pogostosti` prešteje kategorije **skupaj z manjkajočimi** (`manjka`) in jih uredi po velikosti.
`value_counts(dropna=False, normalize=True)` izpiše še deleže.

Poskusite: `"regija"`, `"dohodek_razred"`, `"dualband"`, `"odhod"`.

**Vprašanje:** Je razmerje med strankami, ki so odšle, in tistimi, ki so ostale, uravnoteženo?

In [ ]:
STOLPEC = "regija"            # ← zamenjajte

graf_pogostosti(telekom, STOLPEC).show()
telekom[STOLPEC].value_counts(dropna=False, normalize=True).round(3)   # deleži

<details>
<summary><b>Rešitev (klikni)</b></summary>

`odhod` je skoraj točno 50 : 50. V resnici odide veliko manj strank (nekaj % na mesec) —
nabor je bil **namenoma uravnotežen** za učenje modelov. To moramo vedeti, preden iz podatkov sklepamo o poslovanju.

</details>

## B3. Se stranke, ki odidejo, razlikujejo od tistih, ki ostanejo?
`x="factor(odhod)"` naredi dve škatli, eno za vsako skupino, `fill=` ju pobarva.
`coord_cartesian` pokaže območje od 1. do 99. percentila, da so škatle dobro vidne.
`groupby("odhod")[...].median()` izračuna mediano v vsaki skupini.

Poskusite: `"dni_z_napravo"`, `"cena_naprave"`, `"mesecev_narocnine"`, `"povp_minute"`.
Za violino zamenjajte `geom_boxplot(outlier_shape="", show_legend=False)` z `geom_violin(show_legend=False)`.

**Vprašanje:** Pri katerem stolpcu je razlika med skupinama največja? Je velika?

In [ ]:
STOLPEC = "dni_z_napravo"     # ← zamenjajte

spodaj, zgoraj = telekom[STOLPEC].quantile([0.01, 0.99])
(ggplot(telekom, aes(x="factor(odhod)", y=STOLPEC, fill="factor(odhod)"))
 + geom_boxplot(outlier_shape="", show_legend=False)
 + scale_fill_manual(values=BARVE_ODHOD)              # modra = ostal, oranžna = odšel
 + coord_cartesian(ylim=(spodaj, zgoraj))
 + labs(title=f"{STOLPEC}: ostal (0) / odšel (1)", x="odhod")).show()

telekom.groupby("odhod")[STOLPEC].median()

<details>
<summary><b>Rešitev (klikni)</b></summary>

Stranke, ki so odšle, imajo **dlje isti telefon** (mediana ~ 366 dni proti ~ 310) in **cenejše naprave**.
Razlike so vidne, a majhne — škatli se močno prekrivata. Z enim samim grafom odhoda ne napovemo;
zato bomo kasneje združili več značilk v model.

</details>

## B4. Odnos med dvema numeričnima spremenljivkama
Pri 100 000 strankah vzamemo naključni vzorec (`.sample`), da se točke ne prekrivajo preveč.
`color="factor(odhod)"` obarva skupini.

Poskusite npr. `X = "povp_minute"`, `Y = "povp_mesecni_prihodek"` ali `X = "povp_minute"`, `Y = "povp_minute_6m"`.

In [ ]:
X = "povp_minute"             # ← zamenjajte
Y = "povp_mesecni_prihodek"   # ← zamenjajte

print(f"Korelacija: r = {telekom[X].corr(telekom[Y]):.2f}")
vzorec = telekom.sample(3000, random_state=42)

(ggplot(vzorec, aes(x=X, y=Y, color="factor(odhod)"))
 + geom_point(alpha=0.5, size=1.5)
 + scale_color_manual(values=BARVE_ODHOD)
 + labs(title=f"{Y} glede na {X}", color="odhod"))

## B5. Korelacijska matrika
`select_dtypes("number")` izbere numerične stolpce, `.corr()` izračuna korelacije med vsemi pari,
`graf_korelacij` (iz zvezka A) jih nariše.

**Vprašanje:** Kateri par stolpcev je najmočneje povezan? Zakaj? Kateri stolpec je najbolj povezan z `odhod`?

In [ ]:
kor = telekom.select_dtypes("number").drop(columns=["id_stranke"]).corr()
graf_korelacij(kor)

<details>
<summary><b>Rešitev (klikni)</b></summary>

- `povp_minute` in `povp_minute_6m` (~0,95): to je skoraj isti podatek, merjen v različnem obdobju.
- Z `odhod` so vse korelacije majhne (|r| ≈ 0,1 ali manj): največja sta `dni_z_napravo` (+) in `cena_naprave` (−).

</details>

Preverimo: korelacije z `odhod`, urejene po velikosti (oranžna = pozitivna, modra = negativna):

In [ ]:
odhod_kor = (kor["odhod"].drop("odhod").sort_values()
             .rename_axis("stolpec").reset_index(name="r"))
print(odhod_kor.round(3).to_string(index=False))

(ggplot(odhod_kor, aes(x="stolpec", y="r", fill="r > 0"))
 + geom_col(show_legend=False)
 + scale_fill_manual(values={True: ORANZNA, False: MODRA})
 + scale_x_discrete(limits=odhod_kor["stolpec"].tolist())
 + coord_flip()
 + labs(title="Korelacija stolpcev z 'odhod'", x="", y="r"))

---
### Za hitre: ali korelacija pove vse?
V B5 so bile vse korelacije z `odhod` majhne. Pri `mesecev_narocnine` je r skoraj 0.
Ali to pomeni, da meseci naročnine z odhodom **niso** povezani?

Preverimo drugače: `pd.qcut` razdeli stranke v **10 enako velikih skupin** po vrednosti stolpca
(`(8.0, 11.0]` pomeni: več kot 8 in največ 11 mesecev). Za vsako skupino izračunamo **delež strank, ki so odšle**.
Črtkana črta je povprečje vseh strank (~50 %).

Poskusite še: `"dni_z_napravo"`, `"sprememba_minut"`, `"povp_minute"`.

In [ ]:
STOLPEC = "sprememba_minut"     # ← zamenjajte

print(f"Korelacija z odhod: r = {telekom[STOLPEC].corr(telekom['odhod']):.2f}")
skupine = pd.qcut(telekom[STOLPEC], 10, duplicates="drop", precision=0)    # 10 enako velikih skupin strank
delez = telekom.groupby(skupine, observed=True)["odhod"].mean().mul(100).reset_index(name="odstotek")

(ggplot(delez, aes(x=STOLPEC, y="odstotek"))
 + geom_col(fill=MODRA)
 + geom_hline(yintercept=100 * telekom["odhod"].mean(), linetype="dashed")   # povprečje vseh strank
 + labs(title=f"Delež odhodov po skupinah: {STOLPEC}", x="", y="% strank, ki so odšle")
 + theme(axis_text_x=element_text(rotation=45, ha="right")))

<details>
<summary><b>Rešitev (klikni)</b></summary>

- `mesecev_narocnine`: r = 0,02, a delež odhodov **ni raven**. Pri 6–8 mesecih odide 33 % strank,
  pri 11–12 mesecih 62 %, potem okoli 50 %. Povezava obstaja, le da ni premica —
  **korelacija meri samo linearno povezanost**.
- `dni_z_napravo`: do ~300 dni s telefonom odide okoli 40 % strank, takoj potem 61 %.
- Možna razlaga (**hipoteza**): po približno letu dni se izteče pogodba ali vezava telefona.
  Podatka o pogodbi v tabeli ni — to bi morali preveriti pri podjetju.
- `sprememba_minut` in `povp_minute`: med strankami, ki malo kličejo ali so klicanje močno zmanjšale,
  odide več strank (55–59 %) kot med največjimi uporabniki (44–48 %).

Te stolpce bomo spet srečali pri modelu za napoved odhoda. Modeli, kot so odločitvena drevesa,
sami najdejo meje, kot je "11 mesecev" — tudi tam, kjer je korelacija skoraj 0.

</details>

Prosta celica za vaše grafe: